# Kairos Quickstart — AI Trading Agents

Research-grade AI agents for stocks, ETFs, and crypto — with full decision transparency.

```bash
pip install kairos
```

## 1. Analyze a Stock (AAPL)

Fetches real data via Yahoo Finance, runs the 4-agent pipeline, and shows the decision.

In [ ]:
import asyncio
from kairos.data.providers.yahoofinance import YahooFinanceProvider
from kairos.agents.quant import QuantAgent
from kairos.agents.risk import RiskAgent
from kairos.agents.executor import ExecutorAgent
from kairos.agents.base import AgentContext

async def analyze(ticker: str, days: int = 365):
    print(f"Fetching {ticker} data...")
    provider = YahooFinanceProvider()
    df = await provider.fetch_price_data(ticker, days=days)
    print(f"Loaded {len(df)} bars")
    
    config = {"buy_threshold": 65, "sell_threshold": 35}
    quant = QuantAgent(config)
    qr = await quant.process(AgentContext(input_data={"ohlcv": df}))
    
    risk = RiskAgent(config)
    rr = await risk.process(AgentContext(input_data={"portfolio_value": 10000}))
    
    executor = ExecutorAgent(config)
    er = await executor.process(AgentContext(input_data={
        "quant_output": qr.output,
        "risk_output": rr.output,
        "current_price": float(df["close"].iloc[-1]),
    }))
    
    d = er.output
    print(f"\nDecision: {d['decision']} (confidence: {d['confidence']:.0%})")
    print(f"Rationale: {d['decision_rationale']}")
    print(f"\nTechnical Signals:")
    qo = qr.output
    print(f"  Composite Score: {qo['composite_score']:.0f}/100")
    print(f"  RSI(14): {qo['rsi_14']:.1f}")
    print(f"  Signal: {qo['signal']}")
    return d, qo, rr.output

d, qo, ro = await analyze("AAPL", days=180)
print(f"\nRisk: VaR(95%)={ro['var_95']:.2%}, Kelly={ro['kelly_fraction']:.2f}")

## 2. Write a Custom Strategy

Kairos supports pluggable Strategy classes. Write your own logic:

In [ ]:
import pandas as pd
from kairos.strategies.base import Strategy, StrategyContext, Signal
from kairos.strategies.registry import StrategyRegistry
from kairos.backtesting.engine import WalkForwardEngine
from kairos.data.mock import MockDataProvider

class MACrossover(Strategy):
    """Simple MA crossover strategy."""
    def compute_signal(self, ctx: StrategyContext) -> Signal:
        prices = ctx.prices
        fast = prices.rolling(10).mean()
        slow = prices.rolling(30).mean()
        if len(prices) < 30:
            return Signal.hold()
        if fast.iloc[-2] <= slow.iloc[-2] and fast.iloc[-1] > slow.iloc[-1]:
            return Signal.buy(confidence=0.65, reason="Golden cross")
        elif fast.iloc[-2] >= slow.iloc[-2] and fast.iloc[-1] < slow.iloc[-1]:
            return Signal.sell(confidence=0.6, reason="Death cross")
        return Signal.hold()

# Register and backtest
registry = StrategyRegistry()
registry.register_class("ma_cross", MACrossover)
df = MockDataProvider.generate_price_data(days=365, seed=42)
engine = WalkForwardEngine(df, train_size=90, test_size=30)
result = engine.run(registry.get("ma_cross").agent_config)
print(f"Return: {result['aggregate']['cumulative_return']:.2%}")
print(f"Sharpe: {result['aggregate']['sharpe_ratio']:.2f}")

## 3. Compare Multiple Strategies

In [ ]:
from kairos.statistics.bootstrap import BootstrapSignificanceTest

strategies = ["momentum", "mean_reversion", "ma_cross"]
for name in strategies:
    cfg = registry.get(name)
    result = engine.run(cfg.agent_config)
    agg = result['aggregate']
    sig = BootstrapSignificanceTest(result['all_returns'], n_iterations=200).run() if result['all_returns'] else {}
    print(f"{name:16s} | Return: {agg['cumulative_return']:6.2%} | Sharpe: {agg['sharpe_ratio']:.2f} | p={sig.get('p_value', 1.0):.3f}")

## 4. Generate an HTML Report

In [ ]:
from kairos.reports.html_report import generate_html_report

html = generate_html_report(
    token="AAPL", market_type="stock", strategy_name="momentum",
    decision=d['decision'], confidence=d['confidence'],
    rationale=d['decision_rationale'],
    composite=qo['composite_score'], rsi=qo['rsi_14'],
    signal=qo['signal'], adx=qo['adx_14'], atr=qo.get('atr_14', 0),
    var_95=ro['var_95'], cvar=ro.get('cvar_95', 0),
    kelly=ro['kelly_fraction'], is_safe=ro.get('is_safe', True),
    price=float(df['close'].iloc[-1]), n_bars=len(df),
)
with open("kairos_report.html", "w") as f:
    f.write(html)
print("Report saved to kairos_report.html")

## Next Steps

- `kairos analyze AAPL` — terminal analysis with sparklines
- `kairos backtest --token AAPL` — walk-forward backtesting
- `kairos compare AAPL` — multi-strategy comparison
- `kairos report AAPL` — HTML research report
- Write your own `Strategy` class and register it!